# 🤖 머신러닝 한 바퀴 (붓꽃 데이터)
데이터 준비 → 직접 만든 경사하강법 → scikit-learn 모델 → 평가 → 군집화 순서로 진행합니다.
PyRewind로 실행하면 **학습하면서 가중치와 손실(loss)이 어떻게 변하는지** 단계별로 볼 수 있어요.

## 1. 데이터 불러오기

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, confusion_matrix

In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame
df["품종"] = df["target"].map(dict(enumerate(iris.target_names)))
display(df.head())

In [ ]:
df.shape, df["품종"].value_counts().to_dict()

In [ ]:
summary = df.groupby("품종")[["petal length (cm)", "petal width (cm)"]].mean()
summary

## 2. 직접 만드는 선형회귀 (경사하강법)
꽃잎 길이로 꽃잎 너비를 예측합니다. 반복할 때마다 `w`, `b`, `loss` 가 어떻게 바뀌는지 보세요.

In [ ]:
x = df["petal length (cm)"].to_numpy()
y = df["petal width (cm)"].to_numpy()
w, b = 0.0, 0.0
lr = 0.01
losses = []

In [ ]:
for epoch in range(15):
    pred = w * x + b
    error = pred - y
    loss = float((error ** 2).mean())
    grad_w = float(2 * (error * x).mean())
    grad_b = float(2 * error.mean())
    w = w - lr * grad_w
    b = b - lr * grad_b
    losses.append(round(loss, 4))
print("w =", round(w, 3), "b =", round(b, 3), "마지막 loss =", losses[-1])

In [ ]:
plt.plot(losses)
plt.title("loss")
plt.show()

## 3. scikit-learn 으로 분류하기

In [ ]:
X = df[iris.feature_names]
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
len(X_train), len(X_test)

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [ ]:
model = LogisticRegression(max_iter=200)
model.fit(X_train_s, y_train)
pred = model.predict(X_test_s)
acc = accuracy_score(y_test, pred)
print("로지스틱 회귀 정확도:", round(acc, 3))

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=0)
tree.fit(X_train, y_train)
tree_acc = accuracy_score(y_test, tree.predict(X_test))
importance = dict(zip(iris.feature_names, tree.feature_importances_.round(3)))
importance

In [ ]:
cm = confusion_matrix(y_test, pred)
cm

## 4. 여러 모델 비교하기

In [ ]:
results = {}
for depth in [1, 2, 3, 5]:
    clf = DecisionTreeClassifier(max_depth=depth, random_state=0)
    clf.fit(X_train, y_train)
    results[depth] = round(accuracy_score(y_test, clf.predict(X_test)), 3)
best_depth = max(results, key=results.get)
print("깊이별 정확도:", results, "→ 가장 좋은 깊이:", best_depth)

## 5. 군집화 (정답 없이 묶기)

In [ ]:
kmeans = KMeans(n_clusters=3, n_init=10, random_state=0)
df["군집"] = kmeans.fit_predict(X)
crosstab = pd.crosstab(df["품종"], df["군집"])
crosstab

In [ ]:
plt.scatter(df["petal length (cm)"], df["petal width (cm)"], c=df["군집"])
plt.xlabel("petal length")
plt.ylabel("petal width")
plt.show()